# Lab 3.4 - Amazon SageMaker: Training a model

**Educate edition.** Replaces `en_us/3_4-machinelearning.ipynb`.

## Objectives
* Split data into training, validation and test datasets
* Train an XGBoost model

**Prerequisite:** run Lab 3.1 so `vertebral_column.csv` exists.

## Lab configuration - CHOOSE YOUR TRACK

This notebook runs in one of two modes. Set the flag in the next cell.

| | `USE_MANAGED_SAGEMAKER = False` (**Track B**) | `USE_MANAGED_SAGEMAKER = True` (**Track A**) |
|---|---|---|
| Where training runs | Inside this notebook | A separate managed SageMaker job |
| Extra AWS cost | **$0** | A few cents per job |
| Needs S3 bucket | No | Yes |
| Needs IAM execution role with S3 access | No | Yes |
| Needs `ml.*` training quota | No | **Yes** |
| You learn | The ML concepts | The ML concepts **+ the SageMaker managed workflow** |

**If you are on a $1 budget, or a restricted sandbox account, use
Track B.** It produces the same model and the same numbers. Track A is
what the original AWS Academy lab does, and is worth showing if your
account and budget allow it.

In [ ]:
# ================= LAB CONFIGURATION =================
USE_MANAGED_SAGEMAKER = False    # <-- set True for the managed-job track

# Only used when USE_MANAGED_SAGEMAKER = True.
# These are the smallest instance types that SageMaker supports for each
# role, chosen to keep the cost down.
TRAIN_INSTANCE    = 'ml.m5.large'
ENDPOINT_INSTANCE = 'ml.t2.medium'
TRANSFORM_INSTANCE = 'ml.m5.large'
# =====================================================

import warnings; warnings.simplefilter('ignore')
import pandas as pd, numpy as np, os, json

print('Track:', 'A (managed SageMaker)' if USE_MANAGED_SAGEMAKER
      else 'B (in-notebook, no extra AWS cost)')

## Step 1 - Load and encode the target

XGBoost needs a numeric target. We map the two classes to 0/1.

The convention matters for reading the metrics later:
* `Abnormal` -> **1** (the "positive" class we are detecting)
* `Normal`   -> **0**

In [ ]:
df = pd.read_csv('vertebral_column.csv')

class_map = {'Normal': 0, 'Abnormal': 1}
df['target'] = df['class'].map(class_map)

print(df['class'].value_counts())
print()
print('Encoded target:')
print(df['target'].value_counts())
df.head()

## Step 2 - Put the target in the first column

The built-in SageMaker XGBoost algorithm expects CSV input with:

* the **target in the first column**
* **no header row**
* **no index column**

We build the frame in that layout now, so Track A and Track B use
exactly the same files.

In [ ]:
feature_cols = [c for c in df.columns if c not in ('class', 'target')]
model_df = df[['target'] + feature_cols]

print('Column order:', list(model_df.columns))
model_df.head()

## Step 3 - Split into train / validation / test

We use an 80 / 10 / 10 split.

Two details that matter:

* **`stratify`** keeps the ~68/32 class ratio in every split. Without
  it, a 31-row test set could easily end up with very few `Normal`
  cases and the metrics in Lab 3.6 would be noise.
* **`random_state`** makes the split reproducible, so your numbers
  match across runs and across labs.

We split twice: first off the test set, then split the remainder into
train and validation.

In [ ]:
from sklearn.model_selection import train_test_split

train_val, test = train_test_split(
    model_df, test_size=0.10, random_state=42, stratify=model_df['target'])

train, validation = train_test_split(
    train_val, test_size=1/9, random_state=42, stratify=train_val['target'])

for name, part in [('train', train), ('validation', validation), ('test', test)]:
    pct = len(part) / len(model_df)
    pos = part['target'].mean()
    print(f'{name:11s} {len(part):3d} rows ({pct:.0%})  '
          f'abnormal share {pos:.1%}')

## Step 4 - Write the CSV files

`header=False, index=False` is required by the SageMaker XGBoost
container. We keep the test set's labels in a separate file so we can
score predictions in Lab 3.6.

In [ ]:
train.to_csv('train.csv', header=False, index=False)
validation.to_csv('validation.csv', header=False, index=False)

# test features only (no label) - used for inference in labs 3.5 / 3.6
test[feature_cols].to_csv('test_features.csv', header=False, index=False)
# test labels kept aside for scoring
test[['target']].to_csv('test_labels.csv', header=False, index=False)
# full test set with labels, for convenience
test.to_csv('test.csv', header=False, index=False)

# column names are not in the CSVs, so save them for later labs
with open('feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

for f in ['train.csv','validation.csv','test_features.csv','test_labels.csv']:
    print(f, os.path.getsize(f), 'bytes')

## Step 5 - Train the model

The two tracks diverge here. Run **one** of the two cells below,
depending on your `USE_MANAGED_SAGEMAKER` setting - the code checks the
flag, so you can simply run both and the wrong one will skip itself.

### Track B - train inside the notebook (no extra AWS cost)

This uses the same XGBoost library that the SageMaker container runs
internally, with the same hyperparameters. The model is identical in
kind; only the *place it runs* differs.

`num_boost_round=100` with `early_stopping_rounds=10` means: build up
to 100 trees, but stop early if the validation error has not improved
for 10 consecutive rounds. That prevents overfitting on this small
dataset.

In [ ]:
if not USE_MANAGED_SAGEMAKER:
    import xgboost as xgb

    X_train, y_train = train[feature_cols], train['target']
    X_val,   y_val   = validation[feature_cols], validation['target']

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val,   label=y_val)

    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'error',
        'max_depth': 5,
        'eta': 0.2,
        'subsample': 0.8,
        'min_child_weight': 6,
        'gamma': 4,
        'seed': 42,
    }

    booster = xgb.train(
        params, dtrain,
        num_boost_round=100,
        evals=[(dtrain, 'train'), (dval, 'validation')],
        early_stopping_rounds=10,
        verbose_eval=10,
    )

    booster.save_model('xgboost-model.json')
    print()
    print('Best iteration:', booster.best_iteration)
    print('Validation error at best iteration:', booster.best_score)
    print('Saved model -> xgboost-model.json')
else:
    print('Track A selected - skip this cell, run the next one.')

### Track A - train with a managed SageMaker training job

This is what the original AWS Academy lab does. It:

1. uploads `train.csv` and `validation.csv` to S3
2. launches a **separate** EC2 instance running the AWS XGBoost container
3. writes the model artifact back to S3
4. tears the instance down

**Before running, confirm:**
* your role can read/write the S3 bucket
* you have quota for `ml.m5.large` **training** instances
  (Service Quotas -> Amazon SageMaker -> "ml.m5.large for training job usage")

Expect roughly 3-5 minutes wall-clock, most of which is instance
provisioning. Billed time is short.

In [ ]:
if USE_MANAGED_SAGEMAKER:
    import sagemaker
    from sagemaker.inputs import TrainingInput

    session = sagemaker.Session()
    region  = session.boto_region_name
    bucket  = session.default_bucket()      # sagemaker-<region>-<account-id>
    prefix  = 'mlfoundations/lab3'
    role    = sagemaker.get_execution_role()

    print('Region :', region)
    print('Bucket :', bucket)
    print('Role   :', role)

    train_uri = session.upload_data('train.csv',
                                    bucket=bucket, key_prefix=f'{prefix}/train')
    val_uri   = session.upload_data('validation.csv',
                                    bucket=bucket, key_prefix=f'{prefix}/validation')
    print('Uploaded:', train_uri)
    print('Uploaded:', val_uri)

    image_uri = sagemaker.image_uris.retrieve('xgboost', region, version='1.7-1')
    print('Container:', image_uri)

    estimator = sagemaker.estimator.Estimator(
        image_uri=image_uri,
        role=role,
        instance_count=1,
        instance_type=TRAIN_INSTANCE,
        output_path=f's3://{bucket}/{prefix}/output',
        sagemaker_session=session,
        max_run=1200,                 # hard stop after 20 min - cost guard
    )

    estimator.set_hyperparameters(
        objective='binary:logistic',
        num_round=100,
        max_depth=5,
        eta=0.2,
        subsample=0.8,
        min_child_weight=6,
        gamma=4,
        early_stopping_rounds=10,
    )

    estimator.fit({
        'train':      TrainingInput(train_uri, content_type='csv'),
        'validation': TrainingInput(val_uri,   content_type='csv'),
    })

    print()
    print('Model artifact:', estimator.model_data)

    # save identifiers so labs 3.5 / 3.6 / 3.7 can pick them up
    with open('sm_context.json', 'w') as f:
        json.dump({'bucket': bucket, 'prefix': prefix, 'region': region,
                   'model_data': estimator.model_data,
                   'training_job': estimator.latest_training_job.name,
                   'image_uri': image_uri}, f, indent=2)
    print('Saved sm_context.json')
else:
    print('Track B selected - skip this cell.')

## Step 6 - Quick look at what the model learned

Feature importance tells you which features the trees actually split
on. Compare this to the box plots in Lab 3.2 - the features that
separated the classes visually should rank highly here.

In [ ]:
if not USE_MANAGED_SAGEMAKER:
    import matplotlib.pyplot as plt
    import xgboost as xgb

    imp = booster.get_score(importance_type='gain')
    imp = pd.Series(imp).sort_values()

    imp.plot(kind='barh', figsize=(8, 4),
             title='Feature importance (gain)')
    plt.tight_layout(); plt.show()
    print(imp.sort_values(ascending=False).round(2))
else:
    print('For Track A, view training metrics in the SageMaker console:')
    print('  Training > Training jobs > <your job> > Monitor')

## Conclusion

You have:
* Split the data into training, validation and test sets, stratified on
  the target
* Written CSV files in the layout the SageMaker XGBoost container expects
* Trained an XGBoost model

Files produced, used by the next labs:
`train.csv`, `validation.csv`, `test_features.csv`, `test_labels.csv`,
`test.csv`, `feature_cols.json`, and either `xgboost-model.json`
(Track B) or `sm_context.json` (Track A).

**Cost checkpoint:** if you ran Track A, the training instance has
already terminated automatically. Nothing is still billing except the
notebook instance. **Stop it when you are done.**

Next: `3_5-machinelearning.ipynb`